# Voice Replacer / AI Dubbing — Colab (free GPU)

Run the dubbing pipeline on Colab's **free GPU** — a 9-minute video takes
~5-10 min here instead of 1-2 hours on a CPU.

**Before you start:** `Runtime -> Change runtime type -> Hardware accelerator -> GPU -> Save`.

Then run each cell top to bottom (Shift+Enter).

## 1. Check the GPU is on

In [ ]:
!nvidia-smi -L || echo 'NO GPU: use Runtime -> Change runtime type -> GPU, then re-run.'

## 2. Clone your private repo

You need a **GitHub Personal Access Token** (one-time):
1. Go to https://github.com/settings/tokens?type=beta (fine-grained token)
2. **Generate new token** -> give it a name -> set *Repository access* to
   **Only select repositories -> voice-replacer** -> under *Permissions* set
   **Contents: Read-only** -> Generate.
3. Copy the token (starts with `github_pat_...`) and paste it when asked below.

The token is entered privately (not saved in the notebook).

In [ ]:
from getpass import getpass
import os

USER = 'PrashantRMarathe'
REPO = 'voice-replacer'
token = getpass('Paste your GitHub token: ').strip()

if os.path.isdir(REPO):
    print('Repo already cloned.')
else:
    !git clone https://{token}@github.com/{USER}/{REPO}.git
%cd {REPO}
print('Now in:', os.getcwd())

## 3. Install dependencies (~3-5 min, first run only)

In [ ]:
# Colab already ships a CUDA build of torch; install the rest.
!pip install -q faster-whisper coqui-tts 'transformers<5' ffmpeg-python deep-translator num2words soundfile
!pip install -q openai-whisper
print('Dependencies installed.')

## 4. Configure

On a GPU you can afford a bigger, more accurate transcription model. Edit the
settings below if you like, then run the cell.

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'  # accept XTTS non-commercial license

import config
config.WHISPER_MODEL = 'medium'   # 'large-v3' = best quality, 'small' = fastest

print('Device:', config.DEVICE, '(should be "cuda")')
print('Whisper model:', config.WHISPER_MODEL)

## 5. Upload your video and a reference voice

- **Video:** the MP4 you want to dub / revoice.
- **Reference voice:** a clean 15-30s clip (MP3/WAV) of the voice to clone.

In [ ]:
from google.colab import files

print('Select your VIDEO file...')
video_path = list(files.upload().keys())[0]
print('Select your REFERENCE VOICE file...')
reference_path = list(files.upload().keys())[0]
print('Video:', video_path, '| Reference:', reference_path)

## 6. Dub it

Set `TARGET` to a language code to translate+dub, or leave it `None` to just
replace the voice in the original language.

Codes: en, es, fr, de, it, pt, pl, tr, ru, nl, cs, ar, zh-cn, ja, hu, ko, hi

In [ ]:
import core

TARGET = None   # e.g. 'es' for Spanish, 'hi' for Hindi; None = keep language
SOURCE = 'auto' # or a specific code if auto-detect guesses wrong

out = core.process_video(
    video_path,
    reference_path=reference_path,
    source_lang=SOURCE,
    target_lang=TARGET,
    cb=print,
)
print('\nDONE ->', out)

## 7. Download the result

In [ ]:
from google.colab import files
files.download(out)